# Schema smoke test

Validate that the LLM can answer questions about the corpus given **only** `db/schema.sql` — no embeddings, no graph traversal, no examples. Each fixture question is a separate cell so you can iterate.

What this catches:
- Column names that aren't self-explanatory (the LLM picks the wrong column or invents one).
- Relationships the LLM misreads (e.g. picks legacy `articles.primary_project_id` over the new `article_projects` join).
- Missing affordances (e.g. needs `firm_type` to answer a question and there's no clean way to get it).

What this does **not** catch: scale issues, ambiguous matches across many issues, anything that requires more than one row to surface. Those naturally show up during the Track A reingest.

Re-run the **setup** cell once. After that, edit and re-run question cells freely.

## Setup

In [ ]:
import os
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "db" / "schema.sql").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from core.db import dict_cur, get_conn  # noqa: E402
from core.llm import call_llm  # noqa: E402

# Temporarily bypass Vertex AI: project us-east5 quota for
# anthropic-claude-sonnet-4-5 is 0 pending a quota-increase approval.
# LiteLLM routes "anthropic/..." model strings through Anthropic's
# direct gateway, so this avoids Vertex entirely. Revert to
# core.llm.DEFAULT_MODEL once the Vertex quota lands.
MODEL = "anthropic/claude-sonnet-4-5-20250929"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not set — export it before running this notebook, "
    "or revert MODEL to core.llm.DEFAULT_MODEL once Vertex quota is granted."
)

SCHEMA = (REPO_ROOT / "db" / "schema.sql").read_text()
conn = get_conn()


def _fk_summary(conn) -> str:
    """Pull the FK graph from information_schema. Stays in sync with
    the live DB even if schema.sql parsing would otherwise miss
    constraints added via ALTER TABLE."""
    with dict_cur(conn) as cur:
        cur.execute("""
            SELECT tc.table_name AS from_table,
                   kcu.column_name AS from_col,
                   ccu.table_name AS to_table,
                   ccu.column_name AS to_col
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema   = kcu.table_schema
            JOIN information_schema.constraint_column_usage ccu
              ON tc.constraint_name = ccu.constraint_name
             AND tc.table_schema   = ccu.table_schema
            WHERE tc.constraint_type = 'FOREIGN KEY'
              AND tc.table_schema   = 'public'
            ORDER BY tc.table_name, kcu.column_name
        """)
        rows = cur.fetchall()
    return "\n".join(
        f"  {r['from_table']}.{r['from_col']} -> {r['to_table']}.{r['to_col']}"
        for r in rows
    )


FK_SUMMARY = _fk_summary(conn)


# Few-shot examples: small canonical queries that show how to (a) follow
# FKs to the right column, (b) prefer modern join tables over legacy
# columns, (c) use firm_type instead of guessing role from a string.
EXAMPLES = """\
Q: List every project name and the city it's in.
SQL: SELECT name, city FROM projects LIMIT 50;

Q: Who is the architect for the Salt Palace?
SQL:
SELECT f.name AS architect_name, p.name AS project_name
FROM projects p
JOIN roles r ON r.project_id = p.id
JOIN firms f ON f.id = r.firm_id
WHERE p.name ILIKE '%Salt Palace%'
  AND r.team = 'design'
  AND (r.role ILIKE '%architect%' OR f.firm_type = 'architect')
LIMIT 50;

Q: Which articles cover more than one project?
SQL:
SELECT a.title, array_agg(p.name) AS projects
FROM articles a
JOIN article_projects ap ON ap.article_id = a.id
JOIN projects p ON p.id = ap.project_id
GROUP BY a.id, a.title
HAVING COUNT(*) > 1
LIMIT 50;
"""


ASK_PROMPT = """You are a PostgreSQL expert. Given the schema below, write a SINGLE
Postgres SELECT query that answers the user's question. Return ONLY the SQL —
no markdown fences, no commentary, no leading/trailing prose.

Schema:
{schema}

Foreign keys (authoritative — use these for joins, do not invent columns):
{fk_summary}

Rules:
- SELECT only. No DDL or DML.
- Use ILIKE for case-insensitive name matching.
- LIMIT 50 unless the question requires aggregation over more rows.
- Every table's primary key is named `id`. References from another table
  are named `<referenced_table>_id` on the referencing table. Never invent
  column names; only use columns that appear in the schema above.
- For joins, follow the FK list above as the source of truth. Example:
  `roles.firm_id` references `firms.id`, so `JOIN firms f ON f.id = r.firm_id`
  (NOT `f.firm_id`, which does not exist).
- Prefer modern columns over legacy ones (e.g. join through article_projects
  rather than articles.primary_project_id; use project_sources for
  provenance; use firm_type rather than guessing role from a string).
- If the question is ambiguous, pick the most useful interpretation and
  answer it.

Worked examples:

{examples}
Question: {question}
"""

_SAFE_START = re.compile(r"^\s*(WITH|SELECT)\b", re.IGNORECASE)


def ask(question: str, *, model: str = MODEL, show_sql: bool = True) -> pd.DataFrame:
    """Generate SQL via LLM, run it, return rows as a DataFrame."""
    prompt = ASK_PROMPT.format(
        schema=SCHEMA,
        fk_summary=FK_SUMMARY,
        examples=EXAMPLES,
        question=question,
    )
    raw = call_llm(model, [{"role": "user", "content": prompt}], max_tokens=1024).strip()
    sql = re.sub(r"^```(?:sql)?\s*", "", raw)
    sql = re.sub(r"\s*```$", "", sql).strip()

    if not _SAFE_START.match(sql):
        raise RuntimeError(f"LLM produced non-SELECT SQL:\n{sql}")

    if show_sql:
        print("─── SQL " + "─" * 60)
        print(sql)
        print("─" * 67)

    with dict_cur(conn) as cur:
        cur.execute(sql)
        rows = cur.fetchall()

    return pd.DataFrame(rows) if rows else pd.DataFrame()

### Sanity: what's in the DB right now?

In [ ]:
with dict_cur(conn) as cur:
    counts = {}
    for table in ("issues", "articles", "projects", "firms", "roles",
                  "claims", "quotes", "firm_mentions",
                  "article_projects", "project_sources", "probe_runs"):
        cur.execute(f"SELECT COUNT(*) AS n FROM {table}")
        counts[table] = cur.fetchone()["n"]
pd.Series(counts, name="row_count").to_frame()

## Fixture questions

Eyeball both the SQL and the result for each. A failing fixture is a schema-clarity bug worth fixing before Part 3 starts.

### 1. Basic project listing

In [ ]:
ask("What projects are in the database? Show name, city, typology, and cost.")

### 2. Role lookup (does the LLM use `roles` correctly?)

In [ ]:
ask("Who is the architect for the Delta Sky Club?")

### 3. Full team roster (tests `roles.team` legibility)

In [ ]:
ask("List every firm that worked on the Delta Sky Club, grouped by team (design vs construction vs owner).")

**ISSUE: Wall 2 Wall repeated**

### 4. Claim retrieval (does the LLM filter by `claims.type`?)

In [ ]:
ask("What stat-type claims have been made about any project? Show the claim text and the project name.")

### 5. Quote retrieval with firm filtering

In [ ]:
ask("What quotes are attributed to anyone from HOK?")

### 6. Article ↔ project (does the LLM pick `article_projects` over the legacy column?)

In [ ]:
ask("Which articles cover more than one project? Show the article title and the project names.")

### 7. Provenance (does the LLM understand `project_sources`?)

In [ ]:
ask("Show every project alongside the source it came from (article, public-data feed, etc.).")

### 8. Aggregation across teams

In [ ]:
ask("How many roles per project, broken down by team? Show projects with the most roles first.")

### 9. Cross-row reasoning (firm participation)

In [ ]:
ask("Which firms appear on the most projects, regardless of role? Show firm name and project count.")

### 10. Probe runs (does the LLM understand the cache table?)

In [ ]:
ask("For each article, which probes have been run, and when? Show article title, probe name, and ran_at.")

## Embedding sanity check

Validates that `articles.embedding`, `projects.embedding`, and
`claims.embedding` are usable for the chat agent's `semantic_search`
tool. Embeds a free-text query with the same model that populated the
columns (`text-embedding-3-small`, 1536-dim), runs cosine-distance
nearest-neighbor, prints top-k results.

If the embeddings are healthy, related rows rise to the top of each
query. Sanity-fails fast on model mismatch, NULL columns, or bad
vector data.

Requires `OPENAI_API_KEY` exported.

In [ ]:
import litellm

EMBED_MODEL = "text-embedding-3-small"
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY required for semantic search — same model that "
    "populated the embedding columns via core.embeddings.embed."
)


def embed_query(text: str) -> list[float]:
    """Embed a single query string. Returns a 1536-dim list of floats."""
    resp = litellm.embedding(model=EMBED_MODEL, input=[text])
    return resp.data[0]["embedding"]


_TABLE_COLS = {
    "projects": "id, name, typology, city, state",
    "articles": "id, title, LEFT(COALESCE(summary,''),100) AS summary",
    "claims":   "id, LEFT(text,140) AS text, type",
}


def semantic_search(query: str, *, table: str = "projects", limit: int = 5) -> pd.DataFrame:
    """Nearest-neighbor over `<table>.embedding` using cosine distance.
    `sim` is 1 - distance, so higher = more similar (range -1 to 1)."""
    vec = embed_query(query)
    cols = _TABLE_COLS[table]
    sql = (
        f"SELECT {cols}, 1 - (embedding <=> %s::vector) AS sim "
        f"FROM {table} WHERE embedding IS NOT NULL "
        f"ORDER BY embedding <=> %s::vector LIMIT %s"
    )
    with dict_cur(conn) as cur:
        cur.execute(sql, (vec, vec, limit))
        rows = cur.fetchall()
    return pd.DataFrame(rows)

### Projects: aviation / airport queries

Delta Sky Club at SLC airport should rise to the top.

In [ ]:
semantic_search("airport terminal aviation infrastructure", table="projects", limit=5)

### Projects: water infrastructure in northern Utah

Alpine Aqueduct and Chief Toquer Reservoir should appear.

In [ ]:
semantic_search("water reservoir pipeline civil engineering northern Utah", table="projects", limit=5)

### Articles: luxury hospitality / interiors

Feature articles on high-end hotels, interiors, club lounges.

In [ ]:
semantic_search("luxury hospitality interior design hotel", table="articles", limit=5)

### Claims: schedule and budget milestones

Claims about cost, schedule, or completion milestones.

In [ ]:
semantic_search("on time on budget schedule completion milestone", table="claims", limit=5)

## Free-form

Use this cell for ad-hoc questions while iterating on the schema.

In [ ]:
ask("Tell me what articles mention MHTN.")

In [ ]:
ask("What projects has MHTN been involved in?")

In [ ]:
ask("Has MHTN been involved in healthcare projects?")